## ML source code map

[pipeline.py](../../../src/satellites/ml_classification/pipeline.py) coordinates the five steps below. Each step owns a `libraries/` folder.
Configuration is defined in [shared/config/config.py](../../../src/satellites/ml_classification/shared/config/config.py); code used across steps lives in `shared/`.

| Notebook call | Implementation | Step helpers |
| --- | --- | --- |
| `pipeline.run_check()` | [step_01_check/check.py](../../../src/satellites/ml_classification/step_01_check/check.py) | [libraries/](../../../src/satellites/ml_classification/step_01_check/libraries) |
| `pipeline.run_prepare()` | [step_02_prepare/prepare.py](../../../src/satellites/ml_classification/step_02_prepare/prepare.py) | [libraries/](../../../src/satellites/ml_classification/step_02_prepare/libraries) |
| `pipeline.run_train()` | [step_03_train/train.py](../../../src/satellites/ml_classification/step_03_train/train.py) | [libraries/](../../../src/satellites/ml_classification/step_03_train/libraries) |
| `pipeline.run_evaluate()` | [step_04_evaluate/evaluate.py](../../../src/satellites/ml_classification/step_04_evaluate/evaluate.py) | [libraries/](../../../src/satellites/ml_classification/step_04_evaluate/libraries) |
| `pipeline.run_predict()` | [step_05_predict/predict.py](../../../src/satellites/ml_classification/step_05_predict/predict.py) | [libraries/](../../../src/satellites/ml_classification/step_05_predict/libraries) |

`pipeline.create_reports()` dispatches to each step; `pipeline.recalculate_prediction_quality()` belongs to the predict step.

Source folders use `step_01_check/` through `step_05_predict/`; saved run artifacts still use `01_check/` through `05_predict/`.
See the [source guide](../../../src/satellites/ml_classification/README.md) and [file usage index](../../../src/satellites/ml_classification/FILE_USAGE.md) to trace every implementation and its callers.

In [ ]:
from pathlib import Path
import os

from IPython.display import display
from pathlib import Path
from rasterio import features
import os
import pandas as pd
from satellites.shared.constants import SEED_NUMBER
from satellites.shared.notebook import configure_notebook
configure_notebook(seed=42, tensorflow=True)
import satellites.shared.io as io_l

from satellites.ml_classification.pipeline import GeospatialClassificationPipeline
from satellites.ml_classification.shared.config.config import ClassificationPipelineConfig

In [ ]:
WORK_PATH = Path("C:/work_dir/neuro_2024-2025")

PARCELS_PATH = WORK_PATH / "1_eo_parcel_stats_fill" / "satellite_parcel_time_stats.geoparquet"
EDA_REPORT_PATH = WORK_PATH / "2_eo_results_eda_fill" / "eda_annotated_data.parquet"
PROPOSED_FEATURES_PATH = WORK_PATH / "2_eo_results_eda_fill" / "tables" / "feature_selection" / "feature_selection_proposal.csv"
OUTPUT_PATH = WORK_PATH / "3_eo_ml_classification_fill"

class_column = "crop_category"
parcel_id_column = "parcel_id"


In [ ]:
gdf_parcels = io_l.read_data(str(PARCELS_PATH))
eda_parcels = io_l.read_data(str(EDA_REPORT_PATH))
eda_parcels = eda_parcels[[parcel_id_column, "is_outlier_97_5pct"]]
gdf_parcels = gdf_parcels.merge(eda_parcels, on=parcel_id_column, how="left")
gdf_parcels.head(3)

In [ ]:
df_features = io_l.read_data(str(PROPOSED_FEATURES_PATH))
display(df_features.head())

proposed_features = df_features[df_features["proposed_action"]=="use"]["feature"].tolist()
print(f"Proposed features: {len(proposed_features)}")
proposed_features

In [ ]:
gdf_parcels = gdf_parcels.to_crs(2100)
gdf_parcels["x"] = gdf_parcels.centroid.x
gdf_parcels["y"] = gdf_parcels.centroid.y

base_cols = [parcel_id_column, class_column, "geometry", "data_reliability_score", "geom_shape_complexity_score"]
extra_features = [ x for x in ("elevation_mean_m", "GAIOIKANOT", "x", "y") if x in gdf_parcels.columns]
temporal_features = ("_median", "_sd", "_range",)
selected_indices = ("NDMI", "NDRE", "SAVI", "NDVI", "EVI", "NDWI",)
original_features = [x for x in gdf_parcels.columns if any(keyword in x for keyword in temporal_features) and any(idx in x for idx in selected_indices)]

proposed_features = [x for x in proposed_features if any(keyword in x for keyword in temporal_features)]

gdf_parcels = gdf_parcels[base_cols + original_features + extra_features].copy()
features = original_features + extra_features
# features = proposed_features + extra_features

gdf_parcels = gdf_parcels[gdf_parcels[class_column] != 9999].copy()
print(f"Final features for modeling ({len(features)} features): {features}")
display(gdf_parcels.head(3))

In [ ]:
config = ClassificationPipelineConfig(
    project_dir=OUTPUT_PATH,
    target_column=class_column,
    feature_columns=features,
    id_column=parcel_id_column,

    apply_iqr=False,
    spatial_interpolation_method="nearest",  # None, nearest, idw, or kriging

    cv_folds=7,
    test_size=0.2,
    spatial_split=True,
    spatial_split_grid_size=30,
    spatial_split_method="by_row",
    random_state=SEED_NUMBER,

    scoring_primary="f1_macro",
    cv_ranking_method="score_minus_std",
    selection_type="soft_voting",
    top_voting_models=5,
    max_search_candidates=40,
    prediction_confidence_threshold=0.60,

    interpretability_top_models=7,
    interpretability_include_shap=False,

    optimize_class_probabilities=True,
    probability_multiplier_grid=(0.8, 1.0, 1.2, 1.5, 2.0),
    probability_optimization_iterations=2,
    probability_optimization_max_accuracy_drop=0.02,

    rank_confidence_enabled=True,
    force_retrain_models=True,  # Test the new grids instead of reusing cached models.
    class_reliability_enabled=False,

    label_balancing_method="random_oversample",
    lstm_temporal_statistics=temporal_features,
    selected_models=(
        "bagging",
        "bayesian",
        "decision_tree",
        "extra_trees",
        "random_forest",
        "gradient_boosting",
        "lightgbm",
        "xgboost",
        "support_vector_machine",
        "knn",
        "neural_network",
        # "keras_lstm",
   ),
    max_map_geometries=20000,
    open_html_report=True,
)

# Stronger-regularization experiment: select by validation macro-F1 and per-class recall.
# Only selected_models above are trained; optional model grids are ready if enabled.
# Keep features/splits/balancing fixed for comparison; test no oversampling separately.
# Oversampled duplicates still affect leaf counts and internal early-stopping splits.
# XGBoost/LightGBM wrappers do not pass eval_set: no early_stopping_rounds here.
# LightGBM subsample_freq=1 activates the subsample search.
config.tune_params['logistic_regression'] = {
    'model__C': [0.001, 0.01, 0.1],
    'model__solver': ['lbfgs'],
    'model__penalty': ['l2'],
}

config.tune_params['linear_sgd_classifier'] = {
    'model__alpha': [0.001, 0.01, 0.1],
    'model__l1_ratio': [0.0, 0.15, 0.5],
}

config.tune_params['support_vector_machine'] = {
    'model__C': [0.01, 0.1, 1.0],
    'model__kernel': ['linear', 'rbf'],
    'model__gamma': ['scale', 0.0001, 0.001],
    'model__probability': [True],
}

config.tune_params['random_forest'] = {
    'model__n_estimators': [300, 500],
    'model__max_depth': [3, 5, 8],
    'model__min_samples_split': [20, 40, 80],
    'model__min_samples_leaf': [10, 20, 40],
    'model__max_features': ['sqrt', 0.2],
    'model__bootstrap': [True],
    'model__max_samples': [0.6, 0.8],
    'model__ccp_alpha': [0.0, 0.001, 0.005],
    'model__class_weight': [None],
    'model__criterion': ['gini', 'entropy'],
}

config.tune_params['extra_trees'] = {
    'model__n_estimators': [300, 500],
    'model__max_depth': [3, 5, 8],
    'model__min_samples_split': [20, 40, 80],
    'model__min_samples_leaf': [10, 20, 40],
    'model__max_features': ['sqrt', 0.2],
    'model__bootstrap': [True],
    'model__max_samples': [0.6, 0.8],
    'model__ccp_alpha': [0.0, 0.001, 0.005],
}

config.tune_params['decision_tree'] = {
    'model__max_depth': [2, 3, 5],
    'model__min_samples_split': [20, 40, 80],
    'model__min_samples_leaf': [10, 20, 40],
    'model__criterion': ['gini', 'entropy'],
    'model__max_features': [None, 0.5],
    'model__ccp_alpha': [0.001, 0.005, 0.01],
    'model__splitter': ['best'],
}

config.tune_params['bagging'] = {
    'model__n_estimators': [100, 200],
    'model__max_samples': [0.5, 0.7],
    'model__max_features': [0.5, 0.7],
    'model__estimator__max_depth': [3, 5, 8],
    'model__estimator__min_samples_split': [20, 40, 80],
    'model__estimator__min_samples_leaf': [10, 20, 40],
    'model__estimator__ccp_alpha': [0.0, 0.001, 0.005],
}

config.tune_params['gradient_boosting'] = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.03, 0.05],
    'model__max_depth': [1, 2],
    'model__min_samples_split': [40, 80],
    'model__min_samples_leaf': [20, 40],
    'model__subsample': [0.6, 0.8],
    'model__max_features': ['sqrt', 0.3],
}

config.tune_params['hist_gradient_boosting'] = {
    'model__learning_rate': [0.01, 0.03, 0.05],
    'model__max_iter': [50, 100, 200],
    'model__max_depth': [2, 3],
    'model__max_leaf_nodes': [3, 7],
    'model__min_samples_leaf': [20, 40, 60],
    'model__l2_regularization': [1.0, 10.0, 30.0],
}

config.tune_params['xgboost'] = {
    'model__base_estimator__n_estimators': [50, 100, 200],
    'model__base_estimator__learning_rate': [0.01, 0.03, 0.05],
    'model__base_estimator__max_depth': [1, 2, 3],
    'model__base_estimator__min_child_weight': [5, 10, 20],
    'model__base_estimator__subsample': [0.6, 0.8],
    'model__base_estimator__colsample_bytree': [0.5, 0.7],
    'model__base_estimator__reg_alpha': [0.1, 1.0, 5.0],
    'model__base_estimator__reg_lambda': [5.0, 10.0, 30.0],
    'model__base_estimator__gamma': [0.1, 0.5, 1.0],
}

config.tune_params['lightgbm'] = {
    'model__base_estimator__n_estimators': [50, 100, 200],
    'model__base_estimator__learning_rate': [0.01, 0.03, 0.05],
    'model__base_estimator__num_leaves': [3, 7],
    'model__base_estimator__max_depth': [3, 4],
    'model__base_estimator__min_child_samples': [40, 80],
    'model__base_estimator__subsample': [0.6, 0.8],
    'model__base_estimator__subsample_freq': [1],
    'model__base_estimator__colsample_bytree': [0.5, 0.7],
    'model__base_estimator__reg_alpha': [0.1, 1.0, 5.0],
    'model__base_estimator__reg_lambda': [5.0, 10.0, 30.0],
    'model__base_estimator__min_split_gain': [0.0, 0.1, 0.5],
}

config.tune_params['knn'] = {
    'model__n_neighbors': [15, 25, 40],
    'model__weights': ['uniform'],
    'model__p': [1, 2],
    'model__leaf_size': [30],
    'model__algorithm': ['brute'],
}

config.tune_params['bayesian'] = {
    'model__var_smoothing': [1e-07, 1e-06, 1e-05, 0.0001, 0.001],
}

config.tune_params['neural_network'] = {
    'model__hidden_layer_sizes': [(16,), (32,)],
    'model__alpha': [0.01, 0.1, 1.0, 10.0],
    'model__learning_rate_init': [0.0003, 0.001],
    'model__activation': ['relu', 'tanh'],
    'model__early_stopping': [True],
    'model__validation_fraction': [0.2],
    'model__n_iter_no_change': [10],
}

config.tune_params['tensorflow_neural_network'] = {
    'model__hidden_layer_sizes': [(16,), (32,)],
    'model__activation': ['relu', 'tanh'],
    'model__dropout_rate': [0.3, 0.5],
    'model__use_batch_normalization': [False],
    'model__learning_rate': [0.0003, 0.001],
    'model__batch_size': [32, 64],
    'model__l2_regularization': [0.001, 0.01],
}

config.tune_params['keras_lstm'] = {
    'model__lstm_units_1': [8, 16],
    'model__lstm_units_2': [8],
    'model__dense_units': [8, 16],
    'model__dropout_rate': [0.3, 0.5],
    'model__bidirectional': [False],
    'model__learning_rate': [0.0003, 0.001],
    'model__batch_size': [32, 64],
}

pipeline = GeospatialClassificationPipeline(config)


In [ ]:
summary_check = pipeline.run_check(df=gdf_parcels)
summary_check

In [ ]:
summary_prepare = pipeline.run_prepare()
summary_prepare

In [ ]:
summary_train = pipeline.run_train()
summary_train

In [ ]:
summary_evaluate = pipeline.run_evaluate()
summary_evaluate

In [ ]:
summary_predict = pipeline.run_predict()
summary_predict